# Step 4: Real-time inference and model monitoring

<div style="border: 2px solid #ff9900; border-radius: 8px; padding: 15px; background-color: #fff3e0; margin-bottom: 10px;">
<strong>⚠️ Compatibility Notice:</strong> This notebook has been tested using <strong>SageMaker Distribution Image 4.3.2</strong> and the <strong>SageMaker Python SDK version 3.20.0</strong>.
</div>

**From idea to production in four steps:**
||||
|---|---|---|
|1. |Experiment in a notebook ||
|2. |Scale with SageMaker AI processing jobs and SageMaker SDK ||
|2b.|Automate model governance with EventBridge and Model Cards ||
|3. |Operationalize with ML pipeline, model registry ||
|4. |Add model deployment and monitoring |**<<<< YOU ARE HERE**|

## What you do here

Notebook 03 left a model in the SageMaker Model Registry: the pipeline registered it, staged it, and the governance officer promoted it to `production` and approved it. Nothing is trained in this notebook. You pick up **that** version and take it the rest of the way:

1. **Resolve the deployment candidate** in the Model Registry: the latest version of the registered model, and refuse to deploy it unless it is `production` **and** `Approved`. In a CI/CD setup this check is the `Approved` EventBridge event that starts the deployment.
2. **Validate the serving code in this kernel** before paying for an endpoint: download exactly what the endpoint will download, import the `inference.py` logged next to the model, and drive its four handlers the way the serving container does.
3. **Deploy the version to a real-time endpoint with [data capture](https://docs.aws.amazon.com/sagemaker/latest/dg/model-monitor-data-capture.html) enabled.** The Model Package carries the inference specification, so the deployment needs no image, script or artifact URI: a `Model`, an `EndpointConfig` with a `DataCaptureConfig`, and an `Endpoint`. Data capture writes every request and response the endpoint sees to S3.
4. **Test the endpoint** by scoring the held-out test set in one request, and log the confusion matrix next to the model in MLflow.
5. **Send production traffic** with a unique `InferenceId` per record, and keep the ground-truth labels aside, the way they arrive later in production.
6. **Read the captured data back** from S3 and compare it to the training baseline with [Evidently (open source)](https://www.evidentlyai.com/evidently-oss): **data drift** on the inputs and, once the labels are joined, **model quality**. Metrics and reports are logged to the same MLflow experiment as the training runs.
7. **Automate the check**: the same Evidently logic runs as a SageMaker Processing job inside a SageMaker Pipeline, on a schedule, and publishes an SNS alert when drift or quality crosses a threshold.

![Model monitoring workflow with real-time endpoints](img/arch-sagemaker-inference-predictiveml-monitoring-RealTime-inf.png "Model monitoring workflow with real-time endpoints")

<div class="alert alert-info">
<b>Where the data comes from.</b> The endpoint serves the Model Package version approved in notebook 03. The <b>baseline</b> (the processed feature set without the label) and the <b>test set</b> were written to S3 by the preprocessing job in notebook 01 and by the pipeline in notebook 03; both notebooks <code>%store</code> their locations and the feature names, and this notebook only reads them. Nothing is computed twice.
</div>

<div style="border: 2px solid #ff9900; border-radius: 8px; padding: 15px; background-color: #fff3e0; margin: 10px 0;">
<strong>⚠️ Simulated production data with artificial drift.</strong> The production traffic sent to the endpoint is the held-out test set with a controlled shift applied to two features, so that the drift detection has something to detect. This is for the demonstration only; in production the captured data is real traffic.
</div>

<div style="border: 4px solid #d32f2f; border-radius: 10px; padding: 20px; background-color: #ffebee; margin: 15px 0;">

### ⛔ IMPORTANT: Python 3.12 Required

All notebooks in this workshop **must** run on **Python 3.12**. The cell below will fail with an `AssertionError` if you are using a different Python version.

If you see an error, follow these steps:

1. Open a **Terminal** in JupyterLab (**File → New → Terminal**)
2. Run the setup script:
   ```bash
   cd ~/aim-408-end-to-end-modelops-on-sagemaker-ai
   ./setup-py312.sh
   ```
3. **Refresh your browser** to pick up the new kernel
4. In this notebook, go to **Kernel → Change Kernel → Python 3.12**
5. **Restart the kernel** and re-run from the top

</div>

In [ ]:
import sys

assert sys.version_info[:2] == (3, 12), f"Expected Python 3.12, got {sys.version_info.major}.{sys.version_info.minor}. Please make sure you run on Python 3.12"

## Setup

Restore the variables the earlier notebooks stored: the AWS configuration and MLflow app from notebook 00, the shared experiment and registered model names, the S3 locations of the processed datasets and the feature names from notebooks 01 and 03.

In [ ]:
%store -r

try:
    initialized
except NameError:
    print("+++++++++++++++++++++++++++++++++++++++++++++++++")
    print("[ERROR] YOU HAVE TO RUN 00-start-here notebook   ")
    print("+++++++++++++++++++++++++++++++++++++++++++++++++")

# Everything below is produced by notebooks 01 and 03 and restored by %store -r.
# The datasets are header-less CSVs, so the feature names travel as a stored variable.
required = ["registered_model_name", "experiment_name", "test_s3_url", "baseline_s3_url", "feature_names", "xgboost_image"]
missing = [name for name in required if name not in dir()]
if missing:
    raise NameError(
        f"Missing stored variables {missing}. Run notebooks 01 and 03 first; they %store these values."
    )

In [ ]:
import os
import json
import time
import base64
import importlib.util
from datetime import datetime

import boto3
import numpy as np
import pandas as pd
import mlflow
from mlflow import MlflowClient
from IPython.display import HTML, IFrame, display
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, confusion_matrix

from sagemaker.core.resources import Model, EndpointConfig
from sagemaker.core.shapes import (
    ContainerDefinition,
    ProductionVariant,
    DataCaptureConfig,
    CaptureOption,
    CaptureContentTypeHeader,
)
from sagemaker.core.s3 import S3Downloader
from sagemaker.core.helper.session_helper import Session

from evidently import Report, Dataset, DataDefinition, BinaryClassification
from evidently.presets import DataDriftPreset, DataSummaryPreset, ClassificationPreset

# Registry helpers shared with notebooks 01 and 03.
from governance.approval_gate import latest_model_package, print_lifecycle, mlflow_version_for_package

In [ ]:
sagemaker_session = Session()
sm_client = boto3.client("sagemaker", region_name=region)
runtime_client = boto3.client("sagemaker-runtime", region_name=region)
s3_client = boto3.client("s3", region_name=region)

mlflow.set_tracking_uri(mlflow_arn)
mlflow_client = MlflowClient()
experiment = mlflow.set_experiment(experiment_name)

print("Configuration:")
print(f"  Region:            {region}")
print(f"  S3 bucket/prefix:  s3://{bucket_name}/{bucket_prefix}")
print(f"  IAM role:          {sm_role}")
print(f"  MLflow experiment: {experiment_name}")
print(f"  Registered model:  {registered_model_name}")
print(f"  Features:          {len(feature_names)} (e.g. {feature_names[:4]})")

---
## 1. Resolve the deployment candidate in the Model Registry

The MLflow registered model name is not the name of the SageMaker Model Package Group: the sync appends a short hash suffix when it creates the group. `latest_model_package()` (from [`governance/approval_gate.py`](governance/approval_gate.py), the same helper notebook 03 uses) resolves the group that actually holds the versions and returns its most recent one.

Then the ML engineer's side of the approval gate: the cell re-reads the registry and **refuses to deploy anything that is not `production` and `Approved`**. There is no separate deployment approval anywhere else in this workshop; the registry state is the contract.

What makes the version deployable without any further input is its **inference specification**: the serving image, an `S3Prefix` model data source that points straight at the MLflow artifact store, and the `SAGEMAKER_PROGRAM` environment variable that names the `inference.py` logged next to the model. The cell prints them, together with the provenance the register step and the governance officer recorded on the version.

In [ ]:
latest = latest_model_package(sm_client, registered_model_name, prefix="from-idea-to-prod-model-")
model_package_arn = latest["ModelPackageArn"]

state = print_lifecycle(sm_client, model_package_arn, "Latest version in the registry:")
if not (state["stage"] == "production" and state["approval_status"] == "Approved"):
    raise RuntimeError(
        f"Refusing to deploy {model_package_arn}: stage={state['stage']}, "
        f"approval={state['approval_status']}. Only production/Approved versions are deployed. "
        "Finish the approval gate in notebook 03 first."
    )

described = sm_client.describe_model_package(ModelPackageName=model_package_arn)
container = described["InferenceSpecification"]["Containers"][0]
provenance = described.get("CustomerMetadataProperties", {})

print("\nInference specification (what the endpoint will run):")
print(f"  image:              {container['Image']}")
print(f"  model data source:  {container['ModelDataSource']['S3DataSource']['S3Uri']}")
print(f"  environment:        {container.get('Environment', {})}")
print(f"  instance types:     {described['InferenceSpecification']['SupportedRealtimeInferenceInstanceTypes']}")
print(f"  content types:      {described['InferenceSpecification']['SupportedContentTypes']}")

print("\nProvenance recorded on the version:")
for key in ("RegistrationSource", "RegisteredBy", "Pipeline", "TestAuc", "ApprovedBy", "ApprovedAt"):
    if key in provenance:
        print(f"  {key:<20}{provenance[key]}")
print(f"\nModel card status:    {described.get('ModelCard', {}).get('ModelCardStatus', '(no card)')}")

---
## 2. Validate the serving code in this kernel

Before paying for an endpoint, check that the code it will run actually works. The `S3Prefix` model data source in the inference specification makes SageMaker download **everything** under that prefix to `/opt/ml/model` on the endpoint: the MLflow model files (`MLmodel`, the booster, `conda.yaml`) and the `code/inference.py` the register step logged next to them (by hand in notebook 01; from [`pipeline_steps/inference.py`](pipeline_steps/inference.py) in notebook 03's pipeline, which registered the version deployed here). The BYOC serving stack ([`byoc/serving_app.py`](byoc/serving_app.py)) then imports the script named by `SAGEMAKER_PROGRAM`, calls `model_fn(/opt/ml/model)` once per worker at startup, and dispatches each request to `input_fn` → `predict_fn` → `output_fn`.

You can reproduce that faithfully here: download the same prefix, import the same script the same way, and drive the same four handlers on the test set. No Docker, no endpoint, no cost. A failure here is a failure the endpoint would have surfaced as a `/ping` timeout after several minutes of provisioning.

### Prepare the test data

The held-out test set the preprocessing job wrote to S3: header-less CSV, one record per line, the exact format `inference.py` parses. The feature names restored above give the columns their names again. It is used for the local validation here, for the endpoint test, and as the production traffic later.

In [ ]:
os.makedirs("tmp", exist_ok=True)
S3Downloader.download(f"{test_s3_url}/test_x.csv", "tmp")
S3Downloader.download(f"{test_s3_url}/test_y.csv", "tmp")

test_x = pd.read_csv("tmp/test_x.csv", header=None, names=feature_names)
test_y = pd.read_csv("tmp/test_y.csv", header=None).iloc[:, 0].to_numpy()
print(f"test_x: {test_x.shape} | test_y: {test_y.shape}")
test_x.head(3)

### Download what the endpoint downloads

The model data source of the inference specification is the MLflow artifact location of the logged model. Download that prefix as the endpoint would.

In [ ]:
model_data_s3_uri = container["ModelDataSource"]["S3DataSource"]["S3Uri"]
local_model_dir = os.path.abspath("tmp/model")
os.makedirs(local_model_dir, exist_ok=True)
S3Downloader.download(model_data_s3_uri, local_model_dir)

print(f"Model data source: {model_data_s3_uri}")
print(f"Downloaded to:     {local_model_dir}")
print(f"Contents:          {sorted(os.listdir(local_model_dir))}")
print(f"code/:             {sorted(os.listdir(os.path.join(local_model_dir, 'code')))}")

### Import the script and drive the handlers

`serving_app.py` resolves the script through `SAGEMAKER_PROGRAM` and `SAGEMAKER_SUBMIT_DIRECTORY` (both in the specification's `Environment`), loads it with `importlib`, and calls `model_fn`. Same thing here, then one full request through `input_fn`, `predict_fn` and `output_fn`, with the raw CSV body as bytes, exactly as the HTTP layer hands it over.

In [ ]:
program = container["Environment"]["SAGEMAKER_PROGRAM"]
script_path = os.path.join(local_model_dir, "code", program)

spec = importlib.util.spec_from_file_location("user_inference_module", script_path)
inference_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(inference_module)

local_model = inference_module.model_fn(local_model_dir)
print(f"model_fn returned: {type(local_model).__name__}")

csv_payload = test_x.to_csv(index=False, header=False).encode("utf-8")
data = inference_module.input_fn(csv_payload, "text/csv")
prediction = inference_module.predict_fn(data, local_model)
body, response_content_type = inference_module.output_fn(prediction, "text/csv")
local_predictions = np.array(body.strip().split("\n"), dtype=float)

local_test_auc = roc_auc_score(test_y, local_predictions)
print(f"Parsed input shape:    {data.shape}")
print(f"Response content type: {response_content_type}")
print(f"Local test AUC:        {local_test_auc:.4f}  (TestAuc recorded on the version: {provenance.get('TestAuc', 'n/a')})")

---
## 3. Deploy the approved version with data capture

Deploying from the registry needs no image, no script and no artifact URI at the call site: all of that is in the package's inference specification. A `ContainerDefinition` that names the Model Package is enough. The three SDK v3 resources map one-to-one to the SageMaker API objects: a **`Model`** (what to serve), an **`EndpointConfig`** (how much of it, on what hardware, and what to capture), and an **`Endpoint`** (the running HTTPS service).

The one thing that is new compared to a plain deployment is the **`DataCaptureConfig`** on the endpoint configuration:

- `initial_sampling_percentage` — the share of requests to capture. 100 here so that every request is available to the monitoring; in production with high traffic you sample.
- `capture_options` — capture the `Input`, the `Output`, or both. Drift needs the input, model quality needs the output.
- `capture_content_type_header` — which content types are stored as text; anything else is Base64-encoded.
- `destination_s3_uri` — where the JSON Lines files land, under `<endpoint>/<variant>/YYYY/MM/DD/HH/`.

<div class="alert alert-warning">
<b>SDK gap worth knowing about:</b> <code>DescribeEndpoint</code> omits <code>KmsKeyId</code> from the data capture summary when no key is configured, but the typed <code>Endpoint</code> resource in this SDK version treats that field as required, so <code>Endpoint.create()</code> and <code>Endpoint.get()</code> raise a <code>ValidationError</code> for any capture-enabled endpoint without a KMS key. The <code>Model</code> and <code>EndpointConfig</code> resources are unaffected. The endpoint itself is therefore created, polled and invoked through the <code>boto3</code> client below, which is the same API call without the deserialization.
</div>

<div class="alert alert-info">
This endpoint has no authentication of its own — access is controlled entirely by IAM (<code>sagemaker:InvokeEndpoint</code>) and it is not exposed to the public internet. Keep it that way: do not front it with an unauthenticated proxy, and delete it in the clean-up section when you are done.
</div>

In [ ]:
inference_instance_type = "ml.m5.xlarge"
resource_name = f"from-idea-to-prod-monitored-{time.strftime('%d%H%M%S', time.gmtime())}"
capture_s3_uri = f"s3://{bucket_name}/{bucket_prefix}/data-capture"

deployed_model = Model.create(
    model_name=resource_name,
    primary_container=ContainerDefinition(model_package_name=model_package_arn),
    execution_role_arn=sm_role,
)

endpoint_config = EndpointConfig.create(
    endpoint_config_name=resource_name,
    production_variants=[
        ProductionVariant(
            variant_name="AllTraffic",
            model_name=resource_name,
            initial_instance_count=1,
            instance_type=inference_instance_type,
        )
    ],
    data_capture_config=DataCaptureConfig(
        enable_capture=True,
        initial_sampling_percentage=100,
        destination_s3_uri=capture_s3_uri,
        capture_options=[CaptureOption(capture_mode="Input"), CaptureOption(capture_mode="Output")],
        capture_content_type_header=CaptureContentTypeHeader(csv_content_types=["text/csv"]),
    ),
)

# boto3 for the endpoint: see the SDK gap above.
sm_client.create_endpoint(EndpointName=resource_name, EndpointConfigName=resource_name)
endpoint_name = resource_name

print(f"Creating endpoint {endpoint_name} from {model_package_arn}")
print(f"Data capture destination: {capture_s3_uri}/{endpoint_name}/AllTraffic/")
print("Poll its status in the next cell (this takes a few minutes).")

In [ ]:
# CreateEndpoint returns as soon as the request is accepted, so poll until the
# endpoint reaches a terminal state.
print(f"Polling endpoint: {endpoint_name}")

terminal_states = {"InService", "Failed"}
while True:
    desc = sm_client.describe_endpoint(EndpointName=endpoint_name)
    status = desc["EndpointStatus"]
    print(f"  {time.strftime('%H:%M:%S')} | status={status}")
    if status in terminal_states:
        break
    time.sleep(30)

if status != "InService":
    raise RuntimeError(
        f"Endpoint {endpoint_name} deployment ended with status '{status}': {desc.get('FailureReason')}"
    )

capture = desc["DataCaptureConfig"]
print(f"\nEndpoint {endpoint_name} is InService.")
print(f"Data capture: enabled={capture['EnableCapture']} status={capture['CaptureStatus']} "
      f"sampling={capture['CurrentSamplingPercentage']}% -> {capture['DestinationS3Uri']}")

In [ ]:
display(
    HTML('<b>See the SageMaker <a target="top" href="https://studio-{}.studio.{}.sagemaker.aws/inference-experience/endpoints/details/{}">inference endpoint</a> in the Studio UI</b>'.format(
            domain_id, region, endpoint_name))
)

---
## 4. Test the endpoint

The endpoint is up; verify that it serves the model the registry promised. One request with the whole test set as CSV, the same payload the local validation used, through the SageMaker Runtime API. The AUC has to match the local run to the fourth decimal, and the confusion matrix shows what the model gets right and wrong on real outcomes.

The `inference.py` handlers accept a multi-record CSV body, so the 4,000-odd rows go in a single call and come back as one probability per line.

In [ ]:
response = runtime_client.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="text/csv",
    Accept="text/csv",
    Body=test_x.to_csv(index=False, header=False),
)
predictions = np.array(response["Body"].read().decode("utf-8").strip().split("\n"), dtype=float)

endpoint_test_auc = roc_auc_score(test_y, predictions)
print(f"Endpoint test AUC: {endpoint_test_auc:.4f} | local: {local_test_auc:.4f} | "
      f"max |endpoint - local| = {np.abs(predictions - local_predictions).max():.2e}")
pd.crosstab(index=test_y, columns=np.round(predictions), rownames=["actuals"], colnames=["predictions"])

### Log the endpoint test next to the model in MLflow

The MLflow model version the sync linked to this Model Package points at the training run of the model (`mlflow_version_for_package` matches on the `sagemaker.model_package_arn` tag, rather than assuming version numbers line up). Logging the confusion matrix and the endpoint AUC to that run keeps the evidence next to the model's parameters, metrics and explainability report, where a reviewer looks first.

In [ ]:
def plot_confusion_matrix(cm, class_names, title="Confusion matrix", cmap=plt.cm.Blues, normalize=False):
    if normalize:
        cm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]
    fig, ax = plt.subplots()
    im = ax.imshow(cm, interpolation="nearest", cmap=cmap)
    ax.figure.colorbar(im, ax=ax)
    ax.set(
        xticks=np.arange(cm.shape[1]), yticks=np.arange(cm.shape[0]), ylim=(cm.shape[0] - 0.5, -0.5),
        xticklabels=class_names, yticklabels=class_names, title=title,
        ylabel="Ground truth label", xlabel="Predicted label",
    )
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right", rotation_mode="anchor")
    thresh = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(cm[i, j], ".2f"), ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black")
    fig.tight_layout()
    return ax, fig


ax, fig = plot_confusion_matrix(confusion_matrix(test_y, np.round(predictions)), ["no", "yes"])

In [ ]:
mlflow_version = mlflow_version_for_package(mlflow_client, registered_model_name, model_package_arn)
model_run_id = mlflow_client.get_model_version(registered_model_name, mlflow_version).run_id
print(f"MLflow model version {registered_model_name} v{mlflow_version} -> training run {model_run_id}")

with mlflow.start_run(run_id=model_run_id):
    mlflow.log_metric("endpoint_test_auc", endpoint_test_auc)
    mlflow.log_figure(fig, "endpoint_test_confusion_matrix.png")
    mlflow.set_tags({"endpoint_name": endpoint_name, "model_package_arn": model_package_arn})

presigned_url = sm_client.create_presigned_mlflow_app_url(
    Arn=mlflow_arn, ExpiresInSeconds=60, SessionExpirationDurationInSeconds=1800
)["AuthorizedUrl"]
model_run_link = f"{presigned_url.split('/auth')[0]}/#/experiments/{experiment.experiment_id}/runs/{model_run_id}/artifacts"
display(HTML(f'<b>Open the <a target="_blank" href="{presigned_url}">MLflow UI</a> first, then the <a target="_blank" href="{model_run_link}">artifacts of the model\'s run</a>.</b>'))

---
## 5. Send production traffic

The endpoint works. Now it gets the traffic the monitoring is about: the same test set, sent the way an application would send it, one record per request. Two things mirror how this works in production:

- **Ground truth arrives later.** To measure model quality you need the actual outcome for each prediction, and it is usually known only after the fact: whether the customer did subscribe to the term deposit after the campaign. Here the labels are known already, so they are put aside in their own file, keyed by the same `InferenceId` that is sent with each request. That id is stored in the capture record, and is what joins predictions and outcomes later.
- **The input distribution has moved.** A model in production sees data that drifts away from what it was trained on. To have something to detect, two features are shifted before the data is sent: `campaign` (number of contacts, min-max scaled) is pushed up, and part of the 30–39 age band is moved into the 60–69 band. The one-hot encoding stays consistent, so every record is still a valid input.

In [ ]:
# SIMULATED PRODUCTION DRIFT - for the demonstration only.
rng = np.random.default_rng(42)
production_x = test_x.copy()

# 1. Customers are contacted more often. `campaign` is min-max scaled to [0, 1].
production_x["campaign"] = (production_x["campaign"] * 1.5 + 0.02).clip(upper=1.0)

# 2. An older customer base: 40% of the 30-39 age band moves to 60-69. Both are
#    one-hot columns of the same group, so each record still has exactly one age band.
movers = production_x.index[production_x["age_30-39"] == 1]
movers = rng.choice(movers, size=int(0.4 * len(movers)), replace=False)
production_x.loc[movers, "age_30-39"] = 0
production_x.loc[movers, "age_60-69"] = 1

shifted = ["campaign", "age_30-39", "age_60-69"]
print(pd.DataFrame({"test set": test_x[shifted].mean(), "production traffic": production_x[shifted].mean()}).round(4))

# Ground truth, keyed by the InferenceId sent with each request. Kept apart from
# the traffic, to be joined with the captured predictions later.
inference_ids = [f"inf-{i:06d}" for i in range(len(production_x))]
ground_truth = pd.DataFrame({"inference_id": inference_ids, "target": test_y.astype(int)})
ground_truth.to_csv("tmp/ground_truth.csv", index=False)
print(f"\nGround truth stored for {len(ground_truth)} records")

Each record goes out as its own request with its `InferenceId`, and the probability the endpoint returns is kept, so the live responses can be compared with what data capture stored. The four thousand sequential requests take three to four minutes.

One production detail on the client side: the SageMaker front end occasionally answers a request with a transient `ModelError` that says *"Connection reset by peer ... Please retry"*, even though the container served it. A client retries those, with the same `InferenceId`, so the retry is recognisable in the capture.

In [ ]:
%%time
def invoke(payload, inference_id, attempts=3):
    for attempt in range(1, attempts + 1):
        try:
            response = runtime_client.invoke_endpoint(
                EndpointName=endpoint_name,
                ContentType="text/csv",
                Accept="text/csv",
                Body=payload,
                InferenceId=inference_id,
            )
            return float(response["Body"].read().decode("utf-8").strip())
        except runtime_client.exceptions.ModelError as e:
            if attempt == attempts or "Please retry" not in e.response["Error"]["Message"]:
                raise
            print(f"  transient error on {inference_id}, retrying: {e.response['Error']['Message'][:60]}...")
            time.sleep(2 ** attempt)


live_predictions = np.empty(len(production_x))

print(f"Sending {len(production_x)} requests to {endpoint_name} ...")
for i, (inference_id, row) in enumerate(zip(inference_ids, production_x.itertuples(index=False))):
    live_predictions[i] = invoke(",".join(map(str, row)), inference_id)
    if (i + 1) % 1000 == 0:
        print(f"  {i + 1} requests sent")

print(f"Sent {len(production_x)} requests")
print(f"Test AUC on the (drifted) production traffic: {roc_auc_score(test_y, live_predictions):.4f}")

---
## 6. Read the captured data back from S3

Data capture buffers requests and flushes them to S3 in JSON Lines files under `<destination>/<endpoint>/<variant>/YYYY/MM/DD/HH/`. Each line is one request:

```json
{"captureData": {"endpointInput":  {"observedContentType": "text/csv", "mode": "INPUT",  "data": "0.02,0.96,...", "encoding": "CSV"},
                 "endpointOutput": {"observedContentType": "text/csv", "mode": "OUTPUT", "data": "0.0312",       "encoding": "CSV"}},
 "eventMetadata": {"eventId": "...", "inferenceId": "inf-000010", "inferenceTime": "2025-...Z"},
 "eventVersion": "0"}
```

One record per **request**, whatever its size: the endpoint test in section 4 is a single record whose input and output hold 4,000-odd lines each, and the production traffic is one record per line. The parser below splits multi-line records into one row per line, and pairs each input line with its output line.

Only the requests that carry an `InferenceId` are monitored. That is the production traffic by construction (an application sends the id so that outcomes can be joined later); the endpoint test did not send one and is left out, so the drift analysis compares the baseline against production traffic only.

It can take a minute or two for the last records to land, so the cell keeps re-reading the prefix until every production request sent above has been captured. Data capture stops capturing at high disk usage to protect the inference path, so in production the count does not have to be exact.

In [ ]:
capture_prefix = f"{bucket_prefix}/data-capture/{endpoint_name}/AllTraffic"


def decode(part):
    data = part["data"]
    return base64.b64decode(data).decode("utf-8") if part.get("encoding") == "BASE64" else data


def read_capture(bucket, prefix, feature_names):
    """Parse every capture file under `prefix` into one row per captured input line."""
    rows = []
    paginator = s3_client.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            body = s3_client.get_object(Bucket=bucket, Key=obj["Key"])["Body"].read().decode("utf-8")
            for line in body.strip().split("\n"):
                record = json.loads(line)
                meta = record["eventMetadata"]
                inputs = decode(record["captureData"]["endpointInput"]).strip().split("\n")
                outputs = decode(record["captureData"]["endpointOutput"]).strip().split("\n")
                if len(inputs) != len(outputs):
                    print(f"Skipping record {meta.get('eventId')}: {len(inputs)} inputs vs {len(outputs)} outputs")
                    continue
                for raw_input, raw_output in zip(inputs, outputs):
                    rows.append({
                        "inference_id": meta.get("inferenceId", ""),
                        "inference_time": meta.get("inferenceTime", ""),
                        "prediction_proba": float(raw_output),
                        **dict(zip(feature_names, map(float, raw_input.split(",")))),
                    })
    return pd.DataFrame(rows)


print("Waiting for data capture files in S3 ...")
deadline = time.time() + 600
while True:
    all_captured = read_capture(bucket_name, capture_prefix, feature_names)
    # A retried request can be captured twice under the same InferenceId; keep the last.
    captured_df = (all_captured[all_captured["inference_id"] != ""]
                   .drop_duplicates("inference_id", keep="last").reset_index(drop=True))
    print(f"  {time.strftime('%H:%M:%S')} | {len(captured_df)} of {len(production_x)} production records captured "
          f"(+ {len(all_captured) - len(captured_df)} rows from requests without an InferenceId)")
    if len(captured_df) >= len(production_x) or time.time() > deadline:
        break
    time.sleep(30)

captured_df["prediction"] = (captured_df["prediction_proba"] > 0.5).astype(int)
print(f"\nMonitoring {len(captured_df)} captured production records")
captured_df[["inference_id", "inference_time", "prediction_proba", "prediction"] + feature_names[:3]].head()

In [ ]:
# The captured output is what the endpoint returned: check it against the live responses.
live = pd.DataFrame({"inference_id": inference_ids, "live_proba": live_predictions})
check = captured_df.merge(live, on="inference_id")
max_diff = (check["prediction_proba"] - check["live_proba"]).abs().max()
print(f"{len(check)} captured records matched to live responses; max |captured - live| = {max_diff:.2e}")

---
## 7. Data drift with Evidently

**Data drift** is a change in the distribution of the model's inputs compared to the data it was trained on. It is the earliest signal you can get that a model may be degrading, because it needs no labels.

The **reference** is the baseline the preprocessing job wrote: the full processed feature set, without the label, in the same header-less format as the training data. The **current** data is the captured traffic. Evidently's [`DataDriftPreset`](https://docs.evidentlyai.com/metrics/preset_data_drift) picks a statistical test per column based on its type and size (Wasserstein distance for numerical, Jensen–Shannon for categorical here), reports each column's drift score against its threshold, and the share of drifted columns overall. `DataSummaryPreset` adds descriptive statistics of both datasets side by side.

Everything goes to MLflow, into the **same experiment** as the training runs, tagged with the endpoint and the Model Package it monitors: the HTML and JSON reports as artifacts, and the drift scores as metrics, so runs can be compared over time or turned into alerts.

In [ ]:
S3Downloader.download(f"{baseline_s3_url}/baseline.csv", "tmp")
reference_data = pd.read_csv("tmp/baseline.csv", header=None, names=feature_names)
current_data = captured_df[feature_names]

print(f"Reference (baseline): {reference_data.shape}")
print(f"Current (captured):   {current_data.shape}")

In [ ]:
def log_drift_metrics(report_dict):
    """Log the dataset-level drift share and the score of every drifted column to MLflow."""
    for metric in report_dict.get("metrics", []):
        name, value, config = metric.get("metric_name", ""), metric.get("value"), metric.get("config") or {}
        if name.startswith("DriftedColumnsCount") and isinstance(value, dict):
            mlflow.log_metric("drifted_columns_count", float(value.get("count", 0)))
            mlflow.log_metric("drifted_columns_share", float(value.get("share", 0)))
        elif name.startswith("ValueDrift") and config.get("column") and config.get("threshold") is not None:
            try:
                score = float(value)
            except (TypeError, ValueError):
                continue
            if score > float(config["threshold"]):
                mlflow.log_metric(f"drift_score.{config['column']}", score)


def drifted_columns(report_dict):
    """(column, score, threshold) for every column whose drift score exceeded its threshold."""
    hits = []
    for metric in report_dict.get("metrics", []):
        config = metric.get("config") or {}
        if metric.get("metric_name", "").startswith("ValueDrift") and config.get("threshold") is not None:
            try:
                score = float(metric.get("value"))
            except (TypeError, ValueError):
                continue
            if score > float(config["threshold"]):
                hits.append((config["column"], score, float(config["threshold"])))
    return sorted(hits, key=lambda h: h[1], reverse=True)

In [ ]:
os.makedirs("tmp/reports", exist_ok=True)
timestamp_suffix = datetime.now().strftime("%Y%m%d_%H%M%S")
monitoring_tags = {"endpoint_name": endpoint_name, "model_package_arn": model_package_arn, "monitoring": "evidently"}

with mlflow.start_run(run_name=f"monitoring-data-drift-{timestamp_suffix}", tags=monitoring_tags) as drift_run:
    mlflow.log_params({
        "reference_data": f"{baseline_s3_url}/baseline.csv",
        "reference_size": len(reference_data),
        "current_data": f"{capture_s3_uri}/{endpoint_name}/AllTraffic",
        "current_size": len(current_data),
    })

    data_drift_report = Report(metrics=[DataDriftPreset(), DataSummaryPreset()]).run(
        reference_data=Dataset.from_pandas(reference_data, data_definition=DataDefinition()),
        current_data=Dataset.from_pandas(current_data, data_definition=DataDefinition()),
    )
    data_drift_report_file = f"tmp/reports/data_drift_{timestamp_suffix}"
    data_drift_report.save_html(f"{data_drift_report_file}.html")
    data_drift_report.save_json(f"{data_drift_report_file}.json")
    mlflow.log_artifact(f"{data_drift_report_file}.html", "evidently_data_drift")
    mlflow.log_artifact(f"{data_drift_report_file}.json", "evidently_data_drift")

    drift_results = data_drift_report.dict()
    log_drift_metrics(drift_results)

hits = drifted_columns(drift_results)
print(f"Drifted columns: {len(hits)} of {len(feature_names)}")
for column, score, threshold in hits:
    print(f"  {column:<24}score={score:.4f}  threshold={threshold}")
print(f"\nMLflow run: {drift_run.info.run_name} ({drift_run.info.run_id})")

The two shifted features come out on top, as they should. The report itself is an interactive HTML page: it is logged to MLflow, and can be viewed right here.

In [ ]:
IFrame(src=f"{data_drift_report_file}.html", width=1000, height=700)

---
## 8. Model quality with Evidently

Drift tells you the inputs changed; **model quality** tells you whether the predictions are still right. That needs the ground truth, which is why it was kept aside keyed by `InferenceId`: join it to the captured predictions and you have, per request, the features, the predicted probability, the predicted label and the actual outcome.

Evidently's [`ClassificationPreset`](https://docs.evidentlyai.com/metrics/preset_classification) computes accuracy, precision, recall, F1 and the confusion matrix from that table. A `DataDefinition` with a `BinaryClassification` block tells it which column is the target and which is the prediction. For the bank marketing use case precision is what keeps the campaign cheap (few calls to customers who will not subscribe) and recall is what keeps it effective (few missed subscribers); F1 balances the two.

In [ ]:
ground_truth = pd.read_csv("tmp/ground_truth.csv")
evaluation_data = captured_df.merge(ground_truth, on="inference_id", how="inner")

print(f"Captured predictions: {len(captured_df)} | ground truth: {len(ground_truth)} | joined: {len(evaluation_data)}")
print(pd.crosstab(evaluation_data["target"], evaluation_data["prediction"], rownames=["actual"], colnames=["predicted"]))

In [ ]:
def log_classification_metrics(report_dict):
    """Log every scalar metric of the classification report to MLflow (per-label ones suffixed)."""
    for metric in report_dict.get("metrics", []):
        name = metric.get("metric_name", "").split("(")[0].strip()
        value = metric.get("value")
        if isinstance(value, dict):
            for label, v in value.items():
                mlflow.log_metric(f"{name}_label_{label}", float(v))
        elif value is not None:
            try:
                mlflow.log_metric(name, float(value))
            except (TypeError, ValueError):
                pass


with mlflow.start_run(run_name=f"monitoring-model-quality-{timestamp_suffix}", tags=monitoring_tags) as quality_run:
    mlflow.log_params({"evaluated_records": len(evaluation_data), "ground_truth": "tmp/ground_truth.csv"})

    classification_report = Report(metrics=[ClassificationPreset()]).run(
        reference_data=None,
        current_data=Dataset.from_pandas(
            evaluation_data,
            data_definition=DataDefinition(
                classification=[BinaryClassification(target="target", prediction_labels="prediction", pos_label=1)]
            ),
        ),
    )
    classification_report_file = f"tmp/reports/model_quality_{timestamp_suffix}"
    classification_report.save_html(f"{classification_report_file}.html")
    classification_report.save_json(f"{classification_report_file}.json")
    mlflow.log_artifact(f"{classification_report_file}.html", "evidently_model_quality")
    mlflow.log_artifact(f"{classification_report_file}.json", "evidently_model_quality")

    quality_results = classification_report.dict()
    log_classification_metrics(quality_results)

for metric in quality_results["metrics"]:
    name = metric["metric_name"].split("(")[0]
    if name in ("Accuracy", "Precision", "Recall", "F1Score"):
        print(f"  {name:<12}{float(metric['value']):.4f}")
print(f"\nMLflow run: {quality_run.info.run_name} ({quality_run.info.run_id})")

In [ ]:
IFrame(src=f"{classification_report_file}.html", width=1000, height=600)

### View the monitoring runs in MLflow

Both runs sit in the workshop experiment next to the training runs, tagged with `monitoring = evidently`, the endpoint name and the Model Package ARN. The link below opens the experiment filtered to them; each run carries the drift or quality metrics and the HTML reports under *Artifacts*.

In [ ]:
presigned_url = sm_client.create_presigned_mlflow_app_url(
    Arn=mlflow_arn, ExpiresInSeconds=60, SessionExpirationDurationInSeconds=1800
)["AuthorizedUrl"]
monitoring_runs_link = (
    f"{presigned_url.split('/auth')[0]}/#/experiments/{experiment.experiment_id}/runs"
    f"?workspace=default&searchFilter=tags.monitoring%20%3D%20%22evidently%22"
)
display(HTML(f'<b>Open the <a target="_blank" href="{presigned_url}">MLflow UI</a> first, then the <a target="_blank" href="{monitoring_runs_link}">monitoring runs</a> of the experiment.</b>'))

---
## 9. Automate the check: monitoring as a scheduled SageMaker pipeline

Running Evidently in a notebook is fine for a first look; in production the check has to run without anyone opening a notebook. The same logic goes into a script, the script runs as a **SageMaker Processing job** inside a one-step **SageMaker Pipeline**, and the pipeline runs on a schedule:

- it reads the captured data of the endpoint and the baseline, and computes the drift report; with the ground truth present, the model quality report too;
- it logs the reports and the metrics to MLflow, exactly like sections 7 and 8;
- it publishes an **Amazon SNS** notification when the share of drifted columns, one of the *critical* features, or the F1 score crosses its threshold — for a person to look at, or for a downstream retraining pipeline to react to.

Pipeline parameters make every threshold tunable per execution without redeploying anything.

<div class="alert alert-info">
<b>Permissions.</b> The execution role needs <code>sns:CreateTopic</code> / <code>sns:Publish</code> for the alerts, and <code>scheduler:CreateSchedule</code> plus a trust relationship with <code>scheduler.amazonaws.com</code> if you enable the EventBridge schedule. Workshop accounts may not grant all of these; the cells degrade gracefully (a warning, and the pipeline still runs).
</div>

### An SNS topic for the alerts

Set `ALERT_EMAIL` to receive the alerts by email (the subscription has to be confirmed from the mailbox). Without it the topic is created and the pipeline still publishes to it.

In [ ]:
ALERT_EMAIL = ""  # optional
sns_topic_name = "from-idea-to-prod-monitoring-alerts"

sns_topic_arn = ""
try:
    sns_client = boto3.client("sns", region_name=region)
    sns_topic_arn = sns_client.create_topic(Name=sns_topic_name)["TopicArn"]
    print(f"SNS topic: {sns_topic_arn}")
    if ALERT_EMAIL:
        sns_client.subscribe(TopicArn=sns_topic_arn, Protocol="email", Endpoint=ALERT_EMAIL)
        print(f"Subscription requested for {ALERT_EMAIL} — confirm it from your mailbox.")
except Exception as e:
    print(f"[WARN] Could not set up SNS ({e}). The pipeline will run without alerts.")

### The monitoring script

The processing job reads three inputs: the **baseline** (with a header this time, so the script does not need the feature names passed separately), the endpoint's **data capture** prefix, and the **ground truth** file. The two files are uploaded to a `monitoring/` prefix in S3 now; in production the ground truth would be delivered there by whatever system learns the outcomes.

In [ ]:
monitoring_s3_prefix = f"s3://{bucket_name}/{bucket_prefix}/monitoring"

# upload_data keeps the file name, and the job reads baseline/baseline.csv.
os.makedirs("tmp/monitoring", exist_ok=True)
reference_data.to_csv("tmp/monitoring/baseline.csv", index=False)
monitoring_baseline_s3 = sagemaker_session.upload_data("tmp/monitoring/baseline.csv", bucket_name, f"{bucket_prefix}/monitoring/baseline")
monitoring_ground_truth_s3 = sagemaker_session.upload_data("tmp/ground_truth.csv", bucket_name, f"{bucket_prefix}/monitoring/ground-truth")
endpoint_capture_s3 = f"{capture_s3_uri}/{endpoint_name}/AllTraffic"

print(f"Baseline:     {monitoring_baseline_s3}")
print(f"Ground truth: {monitoring_ground_truth_s3}")
print(f"Capture:      {endpoint_capture_s3}")

The script is the notebook logic of sections 6–8, made parametric. `%%writefile` puts it into a `monitoring/` folder, the same pattern the earlier notebooks used for their processing and training scripts.

In [ ]:
os.makedirs("monitoring", exist_ok=True)

In [ ]:
%%writefile monitoring/evidently_monitor.py
"""Evidently monitoring job: data drift and model quality of a SageMaker endpoint.

Inputs (SageMaker Processing channels):
  /opt/ml/processing/input/baseline/*.csv          reference data, with header
  /opt/ml/processing/input/capture/**/*.jsonl      SageMaker data capture files
  /opt/ml/processing/input/ground_truth/*.csv      inference_id,target (optional)

Logs the reports and metrics to MLflow and publishes an SNS alert when a
threshold is crossed.
"""
import argparse
import base64
import glob
import json
import os
from datetime import datetime, timedelta, timezone

import boto3
import mlflow
import pandas as pd
from evidently import BinaryClassification, Dataset, DataDefinition, Report
from evidently.presets import ClassificationPreset, DataDriftPreset, DataSummaryPreset

INPUT_DIR, OUTPUT_DIR = "/opt/ml/processing/input", "/opt/ml/processing/output"


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--mlflow-tracking-uri", required=True)
    parser.add_argument("--mlflow-experiment", required=True)
    parser.add_argument("--endpoint-name", default="")
    parser.add_argument("--model-package-arn", default="")
    parser.add_argument("--sns-topic-arn", default="")
    parser.add_argument("--drift-threshold", type=float, default=0.1,
                        help="Alert when the share of drifted columns exceeds this value.")
    parser.add_argument("--critical-features", default="",
                        help="Comma-separated features that alert individually when they drift.")
    parser.add_argument("--min-f1", type=float, default=0.0,
                        help="Alert when F1 falls below this value (0 disables; needs ground truth).")
    parser.add_argument("--lookback-days", type=int, default=7,
                        help="Only consider capture records from the last N days (0 = all).")
    return parser.parse_args()


def decode(part):
    data = part["data"]
    return base64.b64decode(data).decode("utf-8") if part.get("encoding") == "BASE64" else data


def load_capture(capture_dir, feature_names, lookback_days):
    """One row per captured input line: inference_id, prediction_proba, prediction, features.

    A capture record is one request, which may hold several CSV lines; they are split
    and paired with the output lines. Only requests carrying an InferenceId are kept:
    that is production traffic by construction, and what ground truth can be joined to."""
    cutoff = datetime.now(timezone.utc) - timedelta(days=lookback_days) if lookback_days > 0 else None
    rows, skipped_old, skipped_no_id = [], 0, 0
    for path in glob.glob(os.path.join(capture_dir, "**", "*.jsonl"), recursive=True):
        with open(path) as f:
            for line in f:
                record = json.loads(line)
                meta = record.get("eventMetadata", {})
                inference_id = meta.get("inferenceId", "")
                if not inference_id:
                    skipped_no_id += 1
                    continue
                if cutoff is not None:
                    try:
                        inference_time = datetime.fromisoformat(meta.get("inferenceTime", "").replace("Z", "+00:00"))
                    except ValueError:
                        inference_time = None
                    if inference_time is not None and inference_time < cutoff:
                        skipped_old += 1
                        continue
                inputs = decode(record["captureData"]["endpointInput"]).strip().split("\n")
                outputs = decode(record["captureData"]["endpointOutput"]).strip().split("\n")
                if len(inputs) != len(outputs):
                    continue
                for raw_input, raw_output in zip(inputs, outputs):
                    rows.append({
                        "inference_id": inference_id,
                        "prediction_proba": float(raw_output),
                        **dict(zip(feature_names, map(float, raw_input.split(",")))),
                    })
    if skipped_old:
        print(f"Skipped {skipped_old} capture records older than {lookback_days} day(s).")
    if skipped_no_id:
        print(f"Skipped {skipped_no_id} capture records without an InferenceId.")
    if not rows:
        return None
    # A retried request can be captured twice under the same InferenceId; keep the last.
    df = pd.DataFrame(rows).drop_duplicates("inference_id", keep="last").reset_index(drop=True)
    df["prediction"] = (df["prediction_proba"] > 0.5).astype(int)
    return df


def drifted_columns(report_dict):
    """(column, score, threshold) for every column whose drift score exceeded its threshold."""
    hits = []
    for metric in report_dict.get("metrics", []):
        config = metric.get("config") or {}
        if metric.get("metric_name", "").startswith("ValueDrift") and config.get("threshold") is not None:
            try:
                score = float(metric.get("value"))
            except (TypeError, ValueError):
                continue
            if score > float(config["threshold"]):
                hits.append((config["column"], score, float(config["threshold"])))
    return sorted(hits, key=lambda h: h[1], reverse=True)


def drifted_share(report_dict):
    for metric in report_dict.get("metrics", []):
        if metric.get("metric_name", "").startswith("DriftedColumnsCount") and isinstance(metric.get("value"), dict):
            return float(metric["value"].get("share", 0.0))
    return 0.0


def scalar_metrics(report_dict, names):
    out = {}
    for metric in report_dict.get("metrics", []):
        name = metric.get("metric_name", "").split("(")[0].strip()
        if name in names:
            try:
                out[name] = float(metric.get("value"))
            except (TypeError, ValueError):
                pass
    return out


def publish_alert(topic_arn, subject, message):
    if not topic_arn or topic_arn == "none":
        print("No SNS topic configured; alert not published:\n" + message)
        return
    try:
        boto3.client("sns", region_name=topic_arn.split(":")[3]).publish(
            TopicArn=topic_arn, Subject=subject[:100], Message=message
        )
        print(f"SNS alert published: {subject}")
    except Exception as e:
        print(f"[WARN] SNS publish failed: {e}")


def main():
    args = parse_args()
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    critical = [c.strip() for c in args.critical_features.split(",") if c.strip()]

    reference = pd.read_csv(sorted(glob.glob(f"{INPUT_DIR}/baseline/*.csv"))[0])
    feature_names = reference.columns.tolist()
    captured = load_capture(f"{INPUT_DIR}/capture", feature_names, args.lookback_days)
    if captured is None:
        print(f"[WARN] No capture records within the last {args.lookback_days} day(s). Nothing to compare.")
        return
    current = captured[feature_names]
    print(f"Reference: {reference.shape} | current: {current.shape}")

    mlflow.set_tracking_uri(args.mlflow_tracking_uri)
    mlflow.set_experiment(args.mlflow_experiment)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    tags = {"monitoring": "evidently", "endpoint_name": args.endpoint_name,
            "model_package_arn": args.model_package_arn, "scheduled": "true"}

    with mlflow.start_run(run_name=f"monitoring-scheduled-{timestamp}", tags=tags) as run:
        mlflow.log_params({
            "reference_size": len(reference), "current_size": len(current),
            "drift_threshold": args.drift_threshold, "critical_features": args.critical_features,
            "min_f1": args.min_f1, "lookback_days": args.lookback_days,
        })

        # ---- data drift
        drift_report = Report(metrics=[DataDriftPreset(), DataSummaryPreset()]).run(
            reference_data=Dataset.from_pandas(reference, data_definition=DataDefinition()),
            current_data=Dataset.from_pandas(current, data_definition=DataDefinition()),
        )
        drift_report.save_html(f"{OUTPUT_DIR}/data_drift_{timestamp}.html")
        drift_report.save_json(f"{OUTPUT_DIR}/data_drift_{timestamp}.json")
        mlflow.log_artifacts(OUTPUT_DIR, "evidently_data_drift")
        drift_dict = drift_report.dict()
        share, hits = drifted_share(drift_dict), drifted_columns(drift_dict)
        mlflow.log_metric("drifted_columns_share", share)
        mlflow.log_metric("drifted_columns_count", len(hits))
        for column, score, _ in hits:
            mlflow.log_metric(f"drift_score.{column}", score)
        print(f"Drifted columns share: {share:.3f} (threshold {args.drift_threshold}); drifted: {[h[0] for h in hits]}")

        # ---- model quality, when ground truth is available
        quality = {}
        ground_truth_files = glob.glob(f"{INPUT_DIR}/ground_truth/*.csv")
        if ground_truth_files:
            ground_truth = pd.concat(pd.read_csv(f) for f in ground_truth_files)
            evaluation = captured.merge(ground_truth, on="inference_id", how="inner")
            if len(evaluation):
                quality_report = Report(metrics=[ClassificationPreset()]).run(
                    reference_data=None,
                    current_data=Dataset.from_pandas(evaluation, data_definition=DataDefinition(
                        classification=[BinaryClassification(target="target", prediction_labels="prediction", pos_label=1)]
                    )),
                )
                quality_report.save_html(f"{OUTPUT_DIR}/model_quality_{timestamp}.html")
                mlflow.log_artifact(f"{OUTPUT_DIR}/model_quality_{timestamp}.html", "evidently_model_quality")
                quality = scalar_metrics(quality_report.dict(), {"Accuracy", "Precision", "Recall", "F1Score"})
                mlflow.log_metrics(quality)
                mlflow.log_param("evaluated_records", len(evaluation))
                print(f"Model quality on {len(evaluation)} labelled records: {quality}")

        # ---- alerting
        reasons = []
        if share > args.drift_threshold:
            reasons.append(f"share of drifted columns {share:.1%} > {args.drift_threshold:.1%}")
        critical_hits = [h for h in hits if h[0] in critical]
        if critical_hits:
            reasons.append("critical features drifted: " + ", ".join(f"{c} ({s:.3f})" for c, s, _ in critical_hits))
        if args.min_f1 > 0 and "F1Score" in quality and quality["F1Score"] < args.min_f1:
            reasons.append(f"F1 {quality['F1Score']:.3f} < {args.min_f1:.3f}")
        mlflow.set_tag("alert", "yes" if reasons else "no")

        if reasons:
            publish_alert(
                args.sns_topic_arn,
                subject=f"[SageMaker] Model monitoring alert: {args.endpoint_name}",
                message=(
                    f"Endpoint:       {args.endpoint_name}\n"
                    f"Model package:  {args.model_package_arn}\n"
                    f"Run:            {timestamp}\n\n"
                    "Triggers:\n  - " + "\n  - ".join(reasons) + "\n\n"
                    "Drifted features (score / threshold):\n  - "
                    + "\n  - ".join(f"{c}: {s:.4f} / {t}" for c, s, t in hits) + "\n\n"
                    f"MLflow experiment: {args.mlflow_experiment}\nMLflow run id:     {run.info.run_id}\n"
                ),
            )
        else:
            print("No alert: all thresholds respected.")


if __name__ == "__main__":
    main()

The processing job runs on the BYOC image from notebook 00, which already ships MLflow and the `sagemaker-mlflow` plugin; Evidently is the only addition, and `FrameworkProcessor` installs a `requirements.txt` found next to the script before running it.

In [ ]:
%%writefile monitoring/requirements.txt
evidently==0.7.23

### Define, register and run the pipeline

One `ProcessingStep` whose `step_args` come from `FrameworkProcessor.run()`, bound to a `PipelineSession` so the job is described rather than started. The pipeline parameters:

| Parameter | Meaning |
|---|---|
| `DriftThreshold` | share of drifted columns that raises an alert |
| `CriticalFeatures` | features that raise an alert individually when they drift, whatever the overall share |
| `MinF1` | when > 0, alert when the F1 score on the labelled records falls below it |
| `LookbackDays` | only capture records from the last N days are compared (0 = everything) |
| `EndpointName`, `SnsTopicArn` | the endpoint under monitoring and where to send alerts |

In [ ]:
from sagemaker.mlops.workflow.pipeline import Pipeline
from sagemaker.mlops.workflow.steps import ProcessingStep, CacheConfig
from sagemaker.core.workflow.parameters import ParameterFloat, ParameterInteger, ParameterString
from sagemaker.core.workflow.pipeline_context import PipelineSession
from sagemaker.core.processing import FrameworkProcessor
from sagemaker.core.shapes import ProcessingInput, ProcessingS3Input, ProcessingOutput, ProcessingS3Output

monitoring_pipeline_name = "from-idea-to-prod-monitoring"

p_drift_threshold = ParameterFloat(name="DriftThreshold", default_value=0.1)
p_critical_features = ParameterString(name="CriticalFeatures", default_value="campaign,pdays,previous")
p_min_f1 = ParameterFloat(name="MinF1", default_value=0.0)
p_lookback_days = ParameterInteger(name="LookbackDays", default_value=7)
p_endpoint_name = ParameterString(name="EndpointName", default_value=endpoint_name)
p_sns_topic_arn = ParameterString(name="SnsTopicArn", default_value=sns_topic_arn or "none")

monitor_processor = FrameworkProcessor(
    image_uri=xgboost_image,
    role=sm_role,
    instance_type="ml.m5.xlarge",
    instance_count=1,
    max_runtime_in_seconds=1200,
    base_job_name="from-idea-to-prod-monitoring",
    command=["python3"],
    sagemaker_session=PipelineSession(),
)

monitor_step = ProcessingStep(
    name="EvidentlyMonitoring",
    step_args=monitor_processor.run(
        code="evidently_monitor.py",
        source_dir="monitoring",
        inputs=[
            ProcessingInput(input_name="baseline", s3_input=ProcessingS3Input(
                s3_uri=monitoring_baseline_s3, local_path="/opt/ml/processing/input/baseline", s3_data_type="S3Prefix")),
            ProcessingInput(input_name="capture", s3_input=ProcessingS3Input(
                s3_uri=endpoint_capture_s3, local_path="/opt/ml/processing/input/capture", s3_data_type="S3Prefix")),
            ProcessingInput(input_name="ground_truth", s3_input=ProcessingS3Input(
                s3_uri=monitoring_ground_truth_s3, local_path="/opt/ml/processing/input/ground_truth", s3_data_type="S3Prefix")),
        ],
        outputs=[
            ProcessingOutput(output_name="reports", s3_output=ProcessingS3Output(
                s3_uri=f"{monitoring_s3_prefix}/reports", local_path="/opt/ml/processing/output", s3_upload_mode="EndOfJob")),
        ],
        arguments=[
            "--mlflow-tracking-uri", mlflow_arn,
            "--mlflow-experiment", experiment_name,
            "--model-package-arn", model_package_arn,
            "--endpoint-name", p_endpoint_name,
            "--sns-topic-arn", p_sns_topic_arn,
            "--drift-threshold", p_drift_threshold.to_string(),
            "--critical-features", p_critical_features,
            "--min-f1", p_min_f1.to_string(),
            "--lookback-days", p_lookback_days.to_string(),
        ],
    ),
    cache_config=CacheConfig(enable_caching=False),
)

monitoring_pipeline = Pipeline(
    name=monitoring_pipeline_name,
    parameters=[p_drift_threshold, p_critical_features, p_min_f1, p_lookback_days, p_endpoint_name, p_sns_topic_arn],
    steps=[monitor_step],
)
monitoring_pipeline.upsert(role_arn=sm_role)
print(f"Pipeline upserted: {monitoring_pipeline_name}")

### Schedule it (optional)

**Amazon EventBridge Scheduler** supports `StartPipelineExecution` as a native target, so a schedule is a single API call with the pipeline parameters inline. Set `ENABLE_SCHEDULE = True` to create a daily run at 08:00 UTC. The role passed to the scheduler must trust `scheduler.amazonaws.com`, which the workshop execution role may not; in that case the cell prints a warning and you start executions manually, as the next cell does.

In [ ]:
ENABLE_SCHEDULE = False
schedule_name = f"{monitoring_pipeline_name}-daily"
schedule_expression = "cron(0 8 * * ? *)"  # every day at 08:00 UTC
account_id = boto3.client("sts").get_caller_identity()["Account"]

if ENABLE_SCHEDULE:
    try:
        boto3.client("scheduler", region_name=region).create_schedule(
            Name=schedule_name,
            ScheduleExpression=schedule_expression,
            FlexibleTimeWindow={"Mode": "OFF"},
            Target={
                "Arn": f"arn:aws:sagemaker:{region}:{account_id}:pipeline/{monitoring_pipeline_name}",
                "RoleArn": sm_role,
                "SageMakerPipelineParameters": {"PipelineParameterList": [
                    {"Name": "EndpointName", "Value": endpoint_name},
                    {"Name": "SnsTopicArn", "Value": sns_topic_arn or "none"},
                ]},
            },
        )
        print(f"Schedule created: {schedule_name} ({schedule_expression})")
    except Exception as e:
        print(f"[WARN] Could not create the schedule ({e}). Start the pipeline manually below.")
else:
    print("Schedule not created (ENABLE_SCHEDULE = False). Start the pipeline manually below.")

### Run it once now

Start an execution to validate the flow end to end. The critical features include `campaign`, which was shifted above, so this run should raise an alert. The job takes about 5 minutes including the instance start-up and the `pip install` of Evidently.

In [ ]:
%%time
execution = monitoring_pipeline.start(parameters={
    "DriftThreshold": 0.1,
    "CriticalFeatures": "campaign,pdays,previous",
    "MinF1": 0.0,
    "LookbackDays": 7,
    "EndpointName": endpoint_name,
    "SnsTopicArn": sns_topic_arn or "none",
})
execution_id = execution.describe()["PipelineExecutionArn"].split("/")[-1]
display(HTML(
    f'<b>See <a target="_blank" href="https://studio-{domain_id}.studio.{region}.sagemaker.aws/'
    f'pipelines/{monitoring_pipeline_name}/executions/{execution_id}/graph">the pipeline execution</a> in the Studio UI</b>'
))

execution.wait()
execution.list_steps()

In [ ]:
# The scheduled run logs to the same experiment, with the tag scheduled=true.
runs = mlflow_client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.monitoring = 'evidently' and tags.scheduled = 'true'",
    order_by=["start_time DESC"],
    max_results=1,
)
if runs:
    run = runs[0]
    print(f"MLflow run: {run.info.run_name} ({run.info.run_id})")
    print(f"  drifted_columns_share: {run.data.metrics.get('drifted_columns_share', 'n/a')}")
    print(f"  F1Score:               {run.data.metrics.get('F1Score', 'n/a')}")
    print(f"  alert:                 {run.data.tags.get('alert', 'n/a')}")
else:
    print("No scheduled monitoring run found; check the processing job logs of the pipeline execution.")

---
## Summary

In this notebook you:

- **Deployed only what governance approved.** Resolved the latest version of the registered model in the SageMaker Model Registry, checked it is `production` and `Approved`, and deployed it with `Model` / `EndpointConfig` / `Endpoint`, from its inference specification alone: no image, script or artifact URI at the call site, and nothing retrained or rebuilt.
- **Validated the serving code first**, by downloading what the endpoint downloads and driving the logged `inference.py` in this kernel exactly as the serving container does, then **tested the endpoint** on the full test set and logged the confusion matrix and AUC to the model's own MLflow run.
- **Turned on data capture** with a `DataCaptureConfig` on the endpoint configuration, and saw every request and response land in S3 as JSON Lines with the `InferenceId` you sent.
- **Kept the ground truth apart**, the way it arrives in production, and joined it back to the captured predictions on the `InferenceId`.
- **Measured data drift** with Evidently's `DataDriftPreset` between the training baseline and the captured traffic, and **model quality** with `ClassificationPreset` once the labels were joined; both logged to the workshop's MLflow experiment as metrics and HTML reports, tagged with the endpoint and the Model Package.
- **Automated the check** as a parametrised SageMaker Pipeline with one Processing step running the same Evidently logic, optionally on an EventBridge schedule, with SNS alerts when drift or quality crosses a threshold.

### Next steps

- **Event-driven monitoring**: start the monitoring pipeline from an S3 event on the data-capture prefix (EventBridge rule → `StartPipelineExecution`) instead of, or in addition to, the schedule.
- **Closed loop**: react to the alert by starting the training pipeline from notebook 03; its register step stages the new version, and the governance officer decides whether it replaces the one in production.
- **More signals**: prediction drift (the distribution of the endpoint's outputs), business KPIs such as the subscription rate, and the fairness metrics from the explain step, computed on the captured traffic.
- **Trends**: compare the monitoring runs in the MLflow UI over time, or build a dashboard on the logged metrics.

## Clean-up

An `InService` endpoint bills per instance-hour whether or not you call it, so delete it. The endpoint configuration and the model cost nothing, but are removed to keep the account tidy, as are the monitoring pipeline, the schedule (if created) and the SNS topic.

The Model Registry, the MLflow experiment (including the monitoring runs) and the S3 data (datasets, capture files, reports) are **kept**.

In [ ]:
for label, delete in (
    ("endpoint", lambda: sm_client.delete_endpoint(EndpointName=endpoint_name)),
    ("endpoint config", lambda: endpoint_config.delete()),
    ("model", lambda: deployed_model.delete()),
    ("pipeline", lambda: monitoring_pipeline.delete()),
):
    try:
        delete()
        print(f"Deleted {label}")
    except Exception as e:
        print(f"Could not delete {label}: {e}")

if ENABLE_SCHEDULE:
    try:
        boto3.client("scheduler", region_name=region).delete_schedule(Name=schedule_name)
        print(f"Deleted schedule {schedule_name}")
    except Exception as e:
        print(f"Could not delete schedule {schedule_name}: {e}")

if sns_topic_arn:
    try:
        sns_client.delete_topic(TopicArn=sns_topic_arn)
        print(f"Deleted SNS topic {sns_topic_arn}")
    except Exception as e:
        print(f"Could not delete SNS topic: {e}")

# Catch anything left over from an earlier run of this notebook.
leftovers = (
    ("endpoint", sm_client.list_endpoints()["Endpoints"], "EndpointName", lambda n: sm_client.delete_endpoint(EndpointName=n)),
    ("endpoint config", sm_client.list_endpoint_configs()["EndpointConfigs"], "EndpointConfigName", lambda n: sm_client.delete_endpoint_config(EndpointConfigName=n)),
    ("model", sm_client.list_models()["Models"], "ModelName", lambda n: sm_client.delete_model(ModelName=n)),
)
for label, items, key, delete in leftovers:
    for item in items:
        name = item[key]
        if name != resource_name and name.startswith("from-idea-to-prod-monitored"):
            try:
                delete(name)
                print(f"Deleted leftover {label} {name}")
            except Exception as e:
                print(f"Could not delete leftover {label} {name}: {e}")

print("\nClean-up complete. MLflow runs, the Model Registry and the S3 data are preserved.")

## Additional resources
- [Capture data from a real-time endpoint](https://docs.aws.amazon.com/sagemaker/latest/dg/model-monitor-data-capture-endpoint.html)
- [Deploy a model from the Model Registry](https://docs.aws.amazon.com/sagemaker/latest/dg/model-registry-deploy.html)
- [Evidently documentation](https://docs.evidentlyai.com/) and [GitHub repository](https://github.com/evidentlyai/evidently)
- [SageMaker Pipelines: schedule executions with EventBridge](https://docs.aws.amazon.com/sagemaker/latest/dg/pipeline-eventbridge.html)
- [Amazon SageMaker AI with MLflow](https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow.html)

# Shutdown kernel

In [ ]:
%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>